# Collimator Mesh Definition

Written by Lev Thomson for python 3.9.25

This script generates a mesh for the thruster and collimator using gmsh's python API. 
The mesh can then be imported into LEoPart-x / FEniCSx for use in plasma simulation. 

# Packages / Libraries:

In [ ]:
import gmsh # For creating mesh
import numpy as np # For adding magnets

# Variables

First, variables are defined for the thruster dimensions, the plasma's Debye length and the dimensions of the mesh. 

In [3]:
run_gui = False # Show the mesh in GMSH GUI

# Thruster / collimator variables:
# TODO note that these are arbitrary values at this stage. 
# Check with team mech re: geometry of thruster.
# Choose magnets as well. 

r_c = 0.0125 # m, = 12.5mm, cathode radius
r_coll = 0.05 # m, = 3cm, collimator radius
r_a = 2*r_c # m, = 25mm, anode radius
r_trig = 0.0005 # m = 0.5mm, trigger radius
kappa = 0.03 # m, = 3cm, cathode-collimator distance
l_c = r_c # m = 12.5mm, length of cathode
l_coll = 0.08 # m, = 10cm, length of collimator
l_a = l_c+kappa # m, = 42.5mm, length of anode
l_trig = l_c * 3/2 # m, = 18.75mm, length of trigger

n_mag = 3 # number of pairs of magnets
h_mag = 0.0025 # m, = 2.5mm, height of magnet
l_mag = l_coll # m, = 10cm, length of magnet
w_mag = 2*h_mag # m, = 5mm, width of magnet

# Plasma variables:
debye_length = 0.005 # TODO

# Model variables:
size_min = 0.001 # TODO
size_max = debye_length

# File variables:

meshname = "collimator.msh" # Mesh will be saved to this file


# Functions

The number of magnets used in the collimator varies, and is specified in n_mag. The function add_magnet_pair (defined below) is called repeatedly when generating the mesh. This adds an evenly-spaced set of n_mag*2 magnets around the inner rim of the collimator. 

In [ ]:
def add_magnet_pair(model:gmsh.model, x:float, y:float, z:float, dx:float, dy:float, dz:float, angle:float, last_tag:int, magnet_surface_list:list, magnet_volume_list:list)->gmsh.model:
    '''
    Adds a pair of magnets at a given point on the collimator-thruster mesh. 
    The magnet is a rectangle defined by a point (x', y', z') and the extents along the x-, y- and z-axes, and is rotated by angle. 
    The second magnet is added diagonally opposite the first.
        Args:
        model: the model to be updated
        x: x-coordinate of one corner of box
        y: y-coordinate of one corner of box
        z: z-coordinate of one corner of box
        dx: x-coordinate of other corner of box
        dy: y-coordinate of other corner of box
        dz: z-coordinate of other corner of box
        angle: (radians) the rectangle is rotated about the x-axis by this angle
        last_tag: the tag of the entity to add the magnets to
        magnet_surface_list: list of dimtags of all surfaces of the magnets
        magnet_volume_list: list of all the tags of the magnet volumes. For defining the magnets as a physical group later. 

    Returns:
        model: updated gmsh model
        last_tag: the tag of the updated model
        magnet_surface_list: list of magnet surfaces by [(dimension=2, tag)] (for defining field later)
    '''
    # Add first magnet
    mag_1_vol = model.occ.addBox(x,-y+1e-6,-z, dx, dy, dz, tag=last_tag+1)
    model.occ.rotate([(3, mag_1_vol)], x+dx/2,0,0, 1,0,0, angle)

    # Add second magnet opposite the first
    mag_2_vol = model.occ.addBox(x,-y+1e-6,-z, dx, dy, dz, tag=last_tag+2)
    model.occ.rotate([(3, mag_2_vol)], x+dx/2,0,0, 1,0,0, angle+np.pi)

    gmsh.model.occ.synchronize() # Cut both at the same time - reduces synchronise calls. 
    
    #model.occ.cut([(3, last_tag+1)], [(3, mag_1_vol)], tag=last_tag+3, removeObject=True, removeTool=False) # Cut first magnet
    #model.occ.cut([(3, last_tag+3)], [(3, mag_2_vol)], tag=last_tag+4, removeObject=True, removeTool=False) # Cut second magnet

    mag_1_surf = model.getBoundary([(3, mag_1_vol)], combined=False, oriented=False)
    mag_2_surf = model.getBoundary([(3, mag_2_vol)], combined=False, oriented=False)
    magnet_surface_list.append(mag_1_surf)
    magnet_surface_list.append(mag_2_surf)
    magnet_volume_list.append(last_tag+1)
    magnet_volume_list.append(last_tag+2)

    gmsh.model.occ.remove([(3, mag_1_vol)], recursive=False) # Then remove the volumes leaving the bounding surfaces intact
    gmsh.model.occ.remove([(3, mag_2_vol)], recursive=False)

    gmsh.model.occ.synchronize()

    # Update tag
    last_tag = last_tag+2
    return model, last_tag, magnet_surface_list, magnet_volume_list

def add_physical_boundary_surface(model:gmsh.model, vol_tag:int|list, phys_tag=int, surf_name=str)->int:
    if type(vol_tag) == int:

        boundary_entities = model.getBoundary([(3, vol_tag)])
        surface_tags = [tag for dim, tag in boundary_entities if dim == 2]

        return gmsh.model.addPhysicalGroup(2, surface_tags, tag=phys_tag, name=surf_name)
    
    elif type(vol_tag) == list:

        surface_tags = []
        for vol in vol_tag:
            boundary_entities = model.getBoundary([(3, vol)])
            for dim, tag in boundary_entities:
                if dim == 2:
                    surface_tags.append(tag)

        return model.addPhysicalGroup(2, surface_tags, tag=phys_tag, name=surf_name)


# Defining a Model Using GMSH

GMSH stores data as a model, which is an object that contains topological entities. 

These entities are:
Order 0 entities: Points
Order 1 entities: Curves, which are defined by multiple points
Order 2 entities: Surfaces, which are defined by multiple curves
Order 3 entities: Volumes, which are defined by multiple surfaces

Each of these entities is identified by a dimtag, which is a tuple with values (dimension, tag).
The dimension is the order of the entity, and the tag is a positive integer that is unique to that entity within that order. (e.g., [(3,1)] is not the same as [(2,1)]. The first is a volume but the second is a surface.) Tags can be defined when the entity is created, or can be generated automatically by GMSH. 

Various operations are available that allow a user to create a complex shape from simpler entities. These include cut and intersect to remove one entity from another, fuse to join entities, and fragment and getBoundary to divide an order-x entity into its constituent parts of order x-1.  

In [ ]:
# ~~~~ Set parameters ~~~~

gmsh.initialize()
gmsh.model.add("collimator") # Add model with name collimator

# ~~~~ Define geometry ~~~~

vol_coll = gmsh.model.occ.addCylinder(l_a,0,0, l_coll,0,0, r_coll, 1) # Collimator cylinder, tag = 1
vol_a = gmsh.model.occ.addCylinder(0,0,0, l_a,0,0, r_a,2) # Anode cylinder, tag = 2
#vol_coll_a = gmsh.model.occ.fuse([(3, vol_coll)], [(3, vol_a)], 3, removeObject=False, removeTool=False)[0] # Fuse collimator to anode. Should look like 2 cylinders stacked on top of each other, tag = 3

# Then remove the cathode from the anode cylinder
vol_c = gmsh.model.occ.addCylinder(0,0,0, l_c,0,0, r_c, 4) # tag = 4
vol_trig = gmsh.model.occ.addCylinder(0,0,0, l_trig,0,0, r_trig, tag=5) # tag = 5
#vol_c_trig = gmsh.model.occ.fuse([(3, vol_c)], [(3, vol_trig)], 6, removeObject=False, removeTool=False)[0] # Fuse cathode and trigger, tag = 6
#vol_thruster = gmsh.model.occ.cut(vol_coll_a, vol_c_trig, 7, removeObject=False, removeTool=False)[0] # Cut the cathode-trigger out of the anode-collimator assembly, tag = 7 
gmsh.model.occ.synchronize()

# Add magnets:
if n_mag == 0:
    pass
elif n_mag > 0:
    last_tag = 5 # Tag of most-recently created entity
    angle_array = np.linspace(0, np.pi, n_mag+1)[0:n_mag] # Generate array of equally spaced magnet pairs around spherical cathode surface
    magnet_surface_list = []
    magnet_volume_list = []
    for loop_idx, angle in enumerate(angle_array):
        gmsh.model, last_tag, magnet_surface_list, magnet_volume_list = add_magnet_pair(gmsh.model, l_a, r_coll, w_mag/2, l_mag, h_mag, w_mag, angle, last_tag, magnet_surface_list, magnet_volume_list)
        
gmsh.model.occ.synchronize()

flat_magnet_surface_tags = [tag for surf in magnet_surface_list for (dim, tag) in surf]
gmsh.model.mesh.embed(2, flat_magnet_surface_tags, 3, vol_coll)

[13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48]


# Defining Physical Groups

GMSH allows users to group entities into "physical groups". This is required to export the .msh file to an .xdmf format for use in FEniCS, but it also makes it possible to define properties for these physical groups. (e.g., all magnets can have the same magnetic properties, which can be different from the collimator wall.)

In [ ]:
# ~~~~ Define physical groups ~~~~~
print(f"magnet surfaces: {magnet_surface_list}")
print(f"magnet volume: {magnet_volume_list}")

phys_col = gmsh.model.addPhysicalGroup(3, [1], 1, "collimator")
phys_a = gmsh.model.addPhysicalGroup(3, [2], 2, "anode")
phys_c = gmsh.model.addPhysicalGroup(3, [4], 4, "cathode")
phys_trig = gmsh.model.addPhysicalGroup(3, [5], 5, "trigger")

phys_surf_col = add_physical_boundary_surface(gmsh.model, 1, 7, "collimator_surface")
phys_surf_a = add_physical_boundary_surface(gmsh.model, 2, 8, "anode_surface")
phys_surf_c = add_physical_boundary_surface(gmsh.model, 4, 9, "cathode_surface")
phys_surf_trig = add_physical_boundary_surface(gmsh.model, 5, 10, "trigger_surface")
phys_surf_mag = gmsh.model.addPhysicalGroup(2, flat_magnet_surface_tags, 11, "magnets_surface")


magnet surfaces: [[(2, 13), (2, 14), (2, 15), (2, 16), (2, 17), (2, 18)], [(2, 19), (2, 20), (2, 21), (2, 22), (2, 23), (2, 24)], [(2, 25), (2, 26), (2, 27), (2, 28), (2, 29), (2, 30)], [(2, 31), (2, 32), (2, 33), (2, 34), (2, 35), (2, 36)], [(2, 37), (2, 38), (2, 39), (2, 40), (2, 41), (2, 42)], [(2, 43), (2, 44), (2, 45), (2, 46), (2, 47), (2, 48)]]
magnet volume: [6, 7, 8, 9, 10, 11]
11


# Defining Fields

After the geometry of the thruster and collimator has been defined, then a field is applied to the model. This allows users to specify the mesh size programatically. For the plasma simulation, the mesh should be smallest where the magnetic field is strong. For this reason, a threshold field is applied to the mesh. This specifies a minimum mesh size, which occurs within distance DistMin to the magnet surfaces. The mesh size then increases up to a maximum size, which occurs at a distance DistMax from the magnet surfaces. 

Distmax is defined as the Debye length. This is the distance over which an ion's magnetic field has an effect, and is the largest size that a mesh cell should be in the plasma simulation. 

The mesh is then generated by GMSH. This can be a computationally intensive process, so it is done in parallel where possible. The resulting mesh is then saved in a .msh file for importing to FEniCSx. 

If the code is run on a computer that supports graphics, then it is displayed in the GMSH gui automatically. 

In [ ]:
# ~~~~ Define mesh field ~~~~
print(f"magnet surfaces: {magnet_surface_list}")

# Get magnet boundary
magnet_array = np.zeros((n_mag*2*6))
for mag_idx, magnet in enumerate(magnet_surface_list):
    for surf_idx, surface in enumerate(magnet):
        magnet_array[6*mag_idx+surf_idx] = surface[1]

distance_field = gmsh.model.mesh.field.add("Distance", 1)
gmsh.model.mesh.field.setNumbers(1, "FacesList", magnet_array)
gmsh.model.mesh.field.setNumber(1, "Sampling", 1000)

threshold_field = gmsh.model.mesh.field.add("Threshold", 2)
gmsh.model.mesh.field.setNumber(2, "InField", distance_field) # Create a mesh field that defines mesh size based on proximity to magnets
gmsh.model.mesh.field.setNumber(2, "SizeMin", size_min)
gmsh.model.mesh.field.setNumber(2, "SizeMax", size_max)
gmsh.model.mesh.field.setNumber(2, "DistMin", 0)
gmsh.model.mesh.field.setNumber(2, "DistMax", 0.05) # 0.5mm

gmsh.model.mesh.field.setAsBackgroundMesh(2)

gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)

# ~~~~ Save & output to GUI ~~~~

gmsh.model.occ.synchronize()

# ~~~~ Parallelize ~~~~

gmsh.option.setNumber("General.NumThreads", 5) # Use multithreading to generate mesh faster
gmsh.option.setNumber("Mesh.Algorithm3D", 4) # Frontal-Delaunay (default. If this causes issues consider changing to 10 - HXT which is a version of Delaunay that is optimised for parallel meshing). 

# ~~~~ Mesh ~~~~

gmsh.model.mesh.generate(3) # Generate in 3d

gmsh.write(meshname)

if run_gui: # Off by default for remote server. 
    gmsh.fltk.run()

gmsh.finalize()

print("Done")

magnet surfaces: [[(2, 13), (2, 14), (2, 15), (2, 16), (2, 17), (2, 18)], [(2, 19), (2, 20), (2, 21), (2, 22), (2, 23), (2, 24)], [(2, 25), (2, 26), (2, 27), (2, 28), (2, 29), (2, 30)], [(2, 31), (2, 32), (2, 33), (2, 34), (2, 35), (2, 36)], [(2, 37), (2, 38), (2, 39), (2, 40), (2, 41), (2, 42)], [(2, 43), (2, 44), (2, 45), (2, 46), (2, 47), (2, 48)]]
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Circle)
Info    : [  0%] Meshing curve 3 (Circle)
Info    : [  0%] Meshing curve 2 (Line)
Info    : [  0%] Meshing curve 4 (Circle)
Info    : [  0%] Meshing curve 5 (Line)
Info    : [  0%] Meshing curve 6 (Circle)
Info    : [  0%] Meshing curve 7 (Circle)
Info    : [  0%] Meshing curve 8 (Line)
Info    : [  0%] Meshing curve 9 (Circle)
Info    : [  0%] Meshing curve 10 (Circle)
Info    : [  0%] Meshing curve 11 (Line)
Info    : [  0%] Meshing curve 12 (Circle)
Info    : [  0%] Meshing curve 13 (Line)
Info    : [  0%] Meshing curve 14 (Line)
Info    : [  0%] Meshing curve 15 (Line)


Info    : Optimization starts (volume = 0.000628205) with worst = 7.13574e-07 / average = 0.651739:
Info    : 0.00 < quality < 0.10 :       500 elements
Info    : 0.10 < quality < 0.20 :      2426 elements
Info    : 0.20 < quality < 0.30 :      9680 elements
Info    : 0.30 < quality < 0.40 :     14330 elements
Info    : 0.40 < quality < 0.50 :      9526 elements
Info    : 0.50 < quality < 0.60 :     21211 elements
Info    : 0.60 < quality < 0.70 :     59498 elements
Info    : 0.70 < quality < 0.80 :     73622 elements
Info    : 0.80 < quality < 0.90 :     27392 elements
Info    : 0.90 < quality < 1.00 :      9885 elements
Info    : 8684 edge swaps, 2468 node relocations (volume = 0.000628205): worst = 0.0215547 / average = 0.66941 (Wall 0.267613s, CPU 0.267815s)
Info    : 10383 edge swaps, 3173 node relocations (volume = 0.000628205): worst = 0.0930106 / average = 0.672392 (Wall 0.354951s, CPU 0.355281s)
Info    : 10685 edge swaps, 3394 node relocations (volume = 0.000628205): worst = 

Info    : Done writing 'collimator.msh'
Done


The mesh is complete. From here, the collimator.msh file can be inspected using the GMSH gui (available at https://www.gmsh.info). 
The mesh is loaded into FEniCSx in the file boundary_conditions.ipynb. 